In [1]:
# Scenario: Customer Support Chatbot Workflow
# Imagine a company wants to build a chatbot that helps customers with quick answers. The workflow is modeled as a graph of states:

# - State Definition (BotState)
# - The chatbot keeps track of:
# - The question asked by the customer.
# - The answer generated.
# - The history of all past questions.
# - Think of this as the chatbot’s notebook where it records the conversation.

# - Nodes (Functions)
# - get_answer:
# When a customer asks, “What are your store hours?”, the chatbot looks at the question and generates a placeholder answer like “Answer to: What are your store hours?”.
# It also adds the question to the history log.
# - format_output:
# Before sending the response back to the customer, the chatbot reformats it into a friendly style:
# “Bot says: Answer to: What are your store hours?”

# - Graph Flow
# - The chatbot starts at the get_answer node (entry point).
# - Once the answer is generated, it flows to the format_output node.
# - Finally, the conversation ends at END, meaning the chatbot has
#  delivered its response.


from langgraph.graph import StateGraph, END
from typing import TypedDict

# 1. Define State
class BotState(TypedDict):
    question: str
    answer: str
    history: list

# 2. Define Nodes (functions)
def get_answer(state: BotState):
    q = state["question"]
    # In real app: call LLM here
    ans = f"Answer to: {q}"
    return {"answer": ans,
            "history": state["history"] + [q]}

def format_output(state: BotState):
    return {"answer": f"Bot says: {state['answer']}"}

# 3. Build the Graph
graph = StateGraph(BotState)
graph.add_node("get_answer", get_answer)
graph.add_node("format", format_output)

# 4. Add Edges
graph.set_entry_point("get_answer")
graph.add_edge("get_answer", "format")
graph.add_edge("format", END)


In [4]:
# Scenario: Customer Support Chatbot (Question-Based)
# Imagine a company has deployed a chatbot that answers customer
#  questions by calling the Groq API. The workflow is modeled as a
#  graph of states, where each customer query flows through nodes until
#   a response is delivered.

# 1. State Definition
# The chatbot maintains a notebook-like state:
# - question → The customer’s query.
# - answer → The response generated by Groq.
# - history → A log of all past questions.

# Step 1: Install
!pip install huggingface-hub langgraph --quiet

# Step 2: Imports
from langgraph.graph import StateGraph, END
from typing import TypedDict
from huggingface_hub import InferenceClient
from google.colab import userdata

# ============================================
# 🔐 LOAD HF API KEY (from Colab secrets)
# ============================================
hf_key = userdata.get('efghi')

if not hf_key:
    raise ValueError("HF API key not found. Save it as 'abcde' in Colab secrets.")

client = InferenceClient(
    provider="auto",
    api_key=hf_key
)

# ============================================
# 🧠 STATE DEFINITION
# ============================================
class BotState(TypedDict):
    question: str
    answer: str
    history: list


# ============================================
# 🤖 NODE 1: GET ANSWER (HF API)
# ============================================
def get_answer(state: BotState):
    q = state["question"]

    try:
        response = client.chat.completions.create(
            model="meta-llama/Meta-Llama-3-8B-Instruct",
            messages=[{"role": "user", "content": q}],
            max_tokens=200
        )

        ans = response.choices[0].message.content

    except Exception as e:
        ans = f"ERROR: {str(e)}"

    return {
        "answer": ans,
        "history": state["history"] + [q]
    }


# ============================================
# 🎨 NODE 2: FORMAT OUTPUT
# ============================================
def format_output(state: BotState):
    return {
        "answer": f"🤖 Bot says: {state['answer']}",
        "history": state["history"]
    }


# ============================================
# 🔗 BUILD GRAPH
# ============================================
graph = StateGraph(BotState)

graph.add_node("get_answer", get_answer)
graph.add_node("format", format_output)

graph.set_entry_point("get_answer")

graph.add_edge("get_answer", "format")
graph.add_edge("format", END)


# ============================================
# ▶️ RUN SYSTEM
# ============================================
if __name__ == "__main__":
    app = graph.compile()

    while True:
        user_input = input("\nAsk something (or 'exit'): ")

        if user_input.lower() == "exit":
            break

        state = {
            "question": user_input,
            "answer": "",
            "history": []
        }

        result = app.invoke(state)

        print("\n✅", result["answer"])


Ask something (or 'exit'): What are your store hours?

✅ 🤖 Bot says: I'm a large language model, so I don't have a physical store. I exist as a digital entity and can be accessed through various devices. I'm available 24/7 to provide information and assist with tasks.

Ask something (or 'exit'): exit


In [5]:
# ============================================
# Scenario: AI-Powered Study Assistant (Flashcard-Based Learning)
# ============================================

# 1. State Definition
# System har study session ke liye ek state maintain karta hai:
# - topic → Kaunsa subject/topic padhna hai (e.g., "Python", "DBMS")
# - question → AI-generated question (flashcard front)
# - correct_answer → Question ka sahi answer
# - user_answer → Student ka diya hua answer
# - result → Evaluation result (Correct / Incorrect)
# - progress → Past attempts ka record (learning history)


# ============================================
# 2. Workflow (Graph of States)
# ============================================

# Har study session ek structured learning flow follow karta hai:

# 🔹 Generate Flashcard Node (AI)
# - AI (LLaMA via HuggingFace) topic ke basis par question + answer generate karta hai
# - State update:
#   question = generated question
#   correct_answer = generated answer


# 🔹 Ask Question Node
# - System student ko question show karta hai
# - Student apna answer enter karta hai
# - State update:
#   user_answer = student input


# 🔹 Evaluate Node (AI)
# - AI student ke answer ko correct answer se compare karta hai
# - Decide karta hai:
#     → "Correct"
#     → "Incorrect"
# - State update:
#   result = evaluation result


# 🔹 Save Progress Node
# - Har attempt ko record kiya jata hai
# - Example log:
#   {
#     question: "...",
#     your_answer: "...",
#     correct_answer: "...",
#     result: "Correct"
#   }
# - State update:
#   progress += new entry


# 🔹 Show Result Node
# - System result display karta hai
# - Correct answer bhi show karta hai
# - Student ko feedback milta hai


# 🔹 End Node
# - Updated progress memory ke saath system next iteration ke liye ready hota hai


# ============================================
# 3. Key Concept
# ============================================

# 👉 Ye system ek "AI-powered learning assistant" hai
# 👉 Flashcards automatically generate hote hain
# 👉 AI evaluation karta hai (self-check learning)
# 👉 Progress tracking se student improvement dekh sakta hai

# Use cases:
# ✔ Exam preparation
# ✔ Self-learning
# ✔ Revision system
# Step 1: Install
!pip install huggingface-hub langgraph --quiet

# Step 2: Imports
from langgraph.graph import StateGraph, END
from typing import TypedDict
from huggingface_hub import InferenceClient
from google.colab import userdata

# ============================================
# 🔐 LOAD HF API KEY (Colab Secret: efghi)
# ============================================
hf_key = userdata.get('efghi')

if not hf_key:
    raise ValueError("HF API key not found. Save it as 'efghi' in Colab secrets.")

client = InferenceClient(
    provider="auto",
    api_key=hf_key
)

# ============================================
# 🧠 STATE
# ============================================
class StudyState(TypedDict):
    topic: str
    question: str
    correct_answer: str
    user_answer: str
    result: str
    progress: list


# ============================================
# 📘 NODE 1: GENERATE FLASHCARD
# ============================================
def generate_flashcard(state: StudyState):
    topic = state["topic"]

    try:
        response = client.chat.completions.create(
            model="meta-llama/Meta-Llama-3-8B-Instruct",
            messages=[{
                "role": "user",
                "content": f"""
                Create 1 flashcard for topic: {topic}

                Format:
                Question: ...
                Answer: ...
                """
            }],
            max_tokens=200
        )

        text = response.choices[0].message.content

        # Simple parsing
        q = text.split("Question:")[-1].split("Answer:")[0].strip()
        a = text.split("Answer:")[-1].strip()

    except Exception as e:
        q = "Error generating question"
        a = str(e)

    return {
        "question": q,
        "correct_answer": a
    }


# ============================================
# ❓ NODE 2: ASK QUESTION
# ============================================
def ask_question(state: StudyState):
    print("\n📘 Question:", state["question"])
    user_ans = input("👉 Your Answer: ")

    return {
        "user_answer": user_ans
    }


# ============================================
# ✅ NODE 3: EVALUATE
# ============================================
def evaluate_answer(state: StudyState):
    try:
        response = client.chat.completions.create(
            model="meta-llama/Meta-Llama-3-8B-Instruct",
            messages=[{
                "role": "user",
                "content": f"""
                Question: {state['question']}
                Correct Answer: {state['correct_answer']}
                Student Answer: {state['user_answer']}

                Is the student correct? Answer only: Correct or Incorrect
                """
            }],
            max_tokens=50
        )

        result = response.choices[0].message.content.strip()

    except Exception as e:
        result = f"Error: {str(e)}"

    return {
        "result": result
    }


# ============================================
# 🧾 NODE 4: SAVE PROGRESS
# ============================================
def save_progress(state: StudyState):
    log = {
        "question": state["question"],
        "your_answer": state["user_answer"],
        "correct_answer": state["correct_answer"],
        "result": state["result"]
    }

    return {
        "progress": state["progress"] + [log]
    }


# ============================================
# 🖨️ NODE 5: SHOW RESULT
# ============================================
def show_result(state: StudyState):
    print("\n📊 Result:", state["result"])
    print("✅ Correct Answer:", state["correct_answer"])

    return state


# ============================================
# 🔗 BUILD GRAPH
# ============================================
graph = StateGraph(StudyState)

graph.add_node("generate", generate_flashcard)
graph.add_node("ask", ask_question)
graph.add_node("evaluate", evaluate_answer)
graph.add_node("save", save_progress)
graph.add_node("show", show_result)

graph.set_entry_point("generate")

graph.add_edge("generate", "ask")
graph.add_edge("ask", "evaluate")
graph.add_edge("evaluate", "save")
graph.add_edge("save", "show")
graph.add_edge("show", END)


# ============================================
# ▶️ RUN SYSTEM
# ============================================
if __name__ == "__main__":
    app = graph.compile()

    progress_memory = []

    while True:
        topic_input = input("\nEnter topic (or 'exit'): ")

        if topic_input.lower() == "exit":
            break

        state = {
            "topic": topic_input,
            "question": "",
            "correct_answer": "",
            "user_answer": "",
            "result": "",
            "progress": progress_memory
        }

        result = app.invoke(state)

        progress_memory = result["progress"]

        print("\n📚 Progress So Far:")
        for i, p in enumerate(progress_memory):
            print(f"{i+1}. {p['question']} → {p['result']}")


Enter topic (or 'exit'): What is photosynthesis?

📘 Question: What is photosynthesis?
👉 Your Answer:  process of making food

📊 Result: Correct
✅ Correct Answer: The process by which plants, algae, and some bacteria convert light energy from the sun into chemical energy in the form of glucose, releasing oxygen as a byproduct.

📚 Progress So Far:
1. What is photosynthesis? → Correct

Enter topic (or 'exit'): exit


In [6]:
# ============================================
# Scenario: AI-Powered Project Management Tracker
# ============================================

# 1. State Definition
# System har project/task ke liye ek state maintain karta hai:
# - task → Kaunsa task chal raha hai (e.g., "Prepare Q1 report")
# - update → Team member ka latest update (e.g., "Draft completed")
# - status → Task ka current status (in progress / completed / blocked)
# - message → System ka confirmation message
# - log → Sabhi past updates ka history record (with timestamp)


# ============================================
# 2. Workflow (Graph of States)
# ============================================

# Har task update ek structured workflow se pass hota hai:

# 🔹 Input Node
# - User (team member) task ka update enter karta hai
# - State update:
#   update = "User input"


# 🔹 Processing Node (HuggingFace LLM)
# - AI model update ko samajhkar task ka status decide karta hai:
#     → "in progress"
#     → "completed"
#     → "blocked"
# - State update:
#   status = "completed" (example)


# 🔹 Response Node
# - System ek confirmation message generate karta hai
# - Example:
#   "Task 'Q1 report' marked as completed"
# - State update:
#   message = confirmation text


# 🔹 Log Node (History Tracking)
# - Har update ko timestamp ke saath store kiya jata hai
# - Example entry:
#   {
#     task: "Q1 report",
#     update: "Draft completed",
#     status: "completed",
#     time: "2026-03-19 10:30:00"
#   }
# - State update:
#   log += new entry


# 🔹 Show History Node
# - System poora project history display karta hai
# - User ko past progress dikhti hai


# 🔹 End Node
# - Updated state return hoti hai with latest log


# ============================================
# 3. Key Concept
# ============================================

# 👉 Ye system ek "AI-assisted project tracker" hai
# 👉 LLM automatically task status classify karta hai
# 👉 Har update ka history maintain hota hai
# 👉 Useful for:
#     ✔ Team collaboration
#     ✔ Progress tracking
#     ✔ Automated reporting
# Step 1: Install
!pip install huggingface-hub langgraph --quiet

# Step 2: Imports
from langgraph.graph import StateGraph, END
from typing import TypedDict
from huggingface_hub import InferenceClient
from google.colab import userdata
from datetime import datetime

# ============================================
# 🔐 LOAD HF API KEY (Colab Secret: efghi)
# ============================================
hf_key = userdata.get('efghi')

if not hf_key:
    raise ValueError("HF API key not found. Save it as 'efghi' in Colab secrets.")

client = InferenceClient(
    provider="auto",
    api_key=hf_key
)

# ============================================
# 🧠 STATE
# ============================================
class ProjectState(TypedDict):
    task: str
    update: str
    status: str
    message: str
    log: list


# ============================================
# 📥 NODE 1: INPUT UPDATE
# ============================================
def input_node(state: ProjectState):
    print("\n📌 Task:", state["task"])
    user_update = input("👉 Enter update: ")

    return {
        "update": user_update
    }


# ============================================
# 🧠 NODE 2: PROCESS UPDATE (HF API)
# ============================================
def process_update(state: ProjectState):
    try:
        response = client.chat.completions.create(
            model="meta-llama/Meta-Llama-3-8B-Instruct",
            messages=[{
                "role": "user",
                "content": f"""
                Task: {state['task']}
                Update: {state['update']}

                Classify task status into one of:
                - in progress
                - completed
                - blocked

                Return only the status.
                """
            }],
            max_tokens=50
        )

        status = response.choices[0].message.content.strip().lower()

    except Exception as e:
        status = f"error: {str(e)}"

    return {
        "status": status
    }


# ============================================
# 📢 NODE 3: RESPONSE
# ============================================
def response_node(state: ProjectState):
    msg = f"✅ Task '{state['task']}' marked as {state['status']}."
    print("\n📢", msg)

    return {
        "message": msg
    }


# ============================================
# 🧾 NODE 4: SAVE LOG
# ============================================
def save_log(state: ProjectState):
    entry = {
        "task": state["task"],
        "update": state["update"],
        "status": state["status"],
        "time": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    return {
        "log": state["log"] + [entry]
    }


# ============================================
# 📊 NODE 5: SHOW HISTORY
# ============================================
def show_log(state: ProjectState):
    print("\n📚 Project Log:")

    for i, entry in enumerate(state["log"]):
        print(f"{i+1}. {entry['task']} → {entry['status']} ({entry['time']})")

    return state


# ============================================
# 🔗 BUILD GRAPH
# ============================================
graph = StateGraph(ProjectState)

graph.add_node("input", input_node)
graph.add_node("process", process_update)
graph.add_node("response", response_node)
graph.add_node("log", save_log)
graph.add_node("show", show_log)

graph.set_entry_point("input")

graph.add_edge("input", "process")
graph.add_edge("process", "response")
graph.add_edge("response", "log")
graph.add_edge("log", "show")
graph.add_edge("show", END)


# ============================================
# ▶️ RUN SYSTEM
# ============================================
if __name__ == "__main__":
    app = graph.compile()

    project_log = []

    while True:
        task_input = input("\nEnter task (or 'exit'): ")

        if task_input.lower() == "exit":
            break

        state = {
            "task": task_input,
            "update": "",
            "status": "",
            "message": "",
            "log": project_log
        }

        result = app.invoke(state)

        project_log = result["log"]


Enter task (or 'exit'):  Q1 financial report

📌 Task:  Q1 financial report
👉 Enter update: draft completed

📢 ✅ Task ' Q1 financial report' marked as completed.

📚 Project Log:
1.  Q1 financial report → completed (2026-03-20 06:25:06)

Enter task (or 'exit'): exit


In [9]:
# ============================================
# Scenario: AI-Powered Customer Support System (LLM-Based Routing)
# ============================================

# 1. State Definition
# The system maintains a ticket-like state for each customer query:
# - query → Customer ka question (e.g., "Why was I charged twice?")
# - category → Query kis department ka hai (billing, tech, general)
# - response → Final reply jo user ko diya jayega


# ============================================
# 2. Workflow (Graph of States)
# ============================================

# Har customer query ek intelligent workflow se pass hoti hai:

# 🔹 Input Node
# - User apna query enter karta hai
# - State update:
#   query = "User ka question"


# 🔹 Router Node (HuggingFace LLM)
# - AI model (LLaMA) query ko analyze karta hai
# - Decide karta hai ki query kis category me aati hai:
#     → "billing"
#     → "tech"
#     → "general"
# - State update:
#   category = "billing" (example)


# 🔹 Conditional Routing
# - Router ke output ke basis pe flow decide hota hai:
#     billing  → billing_agent
#     tech     → tech_agent
#     general  → general_agent


# 🔹 Agent Nodes (Specialized AI Assistants)

# 🧾 Billing Agent
# - Payment, invoice, charges related queries handle karta hai
# - Professional response generate karta hai
# - State update:
#   response = "Billing related answer"

# 🛠️ Tech Agent
# - Errors, bugs, system issues solve karta hai
# - Troubleshooting provide karta hai
# - State update:
#   response = "Technical solution"

# 💬 General Agent
# - Normal/general queries handle karta hai
# - Friendly assistant jaisa reply deta hai
# - State update:
#   response = "General answer"


# 🔹 End Node
# - Final response user ko display hota hai


# ============================================
# 3. Key Concept
# ============================================

# 👉 Ye system ek "AI Router + Multi-Agent Architecture" hai
# 👉 LLM (HuggingFace) smart decision leta hai instead of hardcoded rules
# 👉 Har agent specialized hai → better accuracy & scalability
# Step 1: Install
!pip install huggingface-hub langgraph --quiet

# Step 2: Imports
from langgraph.graph import StateGraph, END
from typing import TypedDict
from huggingface_hub import InferenceClient
from google.colab import userdata

# ============================================
# 🔐 LOAD HF API KEY (Colab Secret: efghi)
# ============================================
hf_key = userdata.get('efghi')

if not hf_key:
    raise ValueError("HF API key not found. Save it as 'efghi' in Colab secrets.")

client = InferenceClient(
    provider="auto",
    api_key=hf_key
)

# ============================================
# 🧠 STATE
# ============================================
class SupportState(TypedDict):
    query: str
    category: str
    response: str


# ============================================
# 🧠 ROUTER NODE (HF API)
# ============================================
def route_query(state: SupportState):
    q = state["query"]

    try:
        response = client.chat.completions.create(
            model="meta-llama/Meta-Llama-3-8B-Instruct",
            messages=[{
                "role": "user",
                "content": f"""
                Classify this customer query into ONE category:
                - billing
                - tech
                - general

                Query: {q}

                Return only one word.
                """
            }],
            max_tokens=10
        )

        category = response.choices[0].message.content.strip().lower()

    except Exception as e:
        category = "general"

    # The router node should return a dictionary to update the state
    return {"category": category}


# ============================================
# 🎯 BILLING AGENT
# ============================================
def billing_agent(state):
    q = state["query"]

    try:
        response = client.chat.completions.create(
            model="meta-llama/Meta-Llama-3-8B-Instruct",
            messages=[{
                "role": "user",
                "content": f"""
                You are billing support.
                Answer the query professionally:

                {q}
                """
            }],
            max_tokens=150
        )

        ans = response.choices[0].message.content

    except Exception as e:
        ans = f"Billing error: {str(e)}"

    return {"response": ans}


# ============================================
# 🛠️ TECH AGENT
# ============================================
def tech_agent(state):
    q = state["query"]

    try:
        response = client.chat.completions.create(
            model="meta-llama/Meta-Llama-3-8B-Instruct",
            messages=[{
                "role": "user",
                "content": f"""
                You are technical support.
                Help solve the issue:

                {q}
                """
            }],
            max_tokens=150
        )

        ans = response.choices[0].message.content

    except Exception as e:
        ans = f"Tech error: {str(e)}"

    return {"response": ans}


# ============================================
# 💭 GENERAL AG💭 GENERAL AGENT
# ============================================
def general_agent(state):
    q = state["query"]

    try:
        response = client.chat.completions.create(
            model="meta-llama/Meta-Llama-3-8B-Instruct",
            messages=[{
                "role": "user",
                "content": f"""
                You are a helpful assistant.
                Answer the query:

                {q}
                """
            }],
            max_tokens=150
        )

        ans = response.choices[0].message.content

    except Exception as e:
        ans = f"General error: {str(e)}"

    return {"response": ans}


# ============================================
# 🔗 BUILD GRAPH
# ============================================
graph = StateGraph(SupportState)

# Add the router function as a node
graph.add_node("router", route_query)
graph.add_node("billing_agent", billing_agent)
graph.add_node("tech_agent", tech_agent)
graph.add_node("general_agent", general_agent)

graph.set_entry_point("router")

graph.add_conditional_edges(
    "router",
    # The condition should now extract the category from the updated state
    lambda state: state["category"],
    {
        "billing": "billing_agent",
        "tech": "tech_agent",
        "general": "general_agent"
    }
)

graph.add_edge("billing_agent", END)
graph.add_edge("tech_agent", END)
graph.add_edge("general_agent", END)


# ============================================
# ▶️ RUN SYSTEM
# ============================================
if __name__ == "__main__":
    app = graph.compile()

    while True:
        user_input = input("\nEnter query (or 'exit'): ")

        if user_input.lower() == "exit":
            break

        state = {
            "query": user_input,
            "category": "",
            "response": ""
        }

        result = app.invoke(state)

        print("\n✅ Response:\n", result["response"])


Enter query (or 'exit'): I have an issue with payment not going through

✅ Response:
 I'm happy to assist you with your payment issue. 

To better understand the problem, could you please provide me with some additional information? 

1. What type of payment method were you attempting to use (e.g., credit card, PayPal, bank transfer)?
2. What was the exact error message you received when the payment failed?
3. Did you receive any confirmation emails or notifications regarding the payment attempt?
4. Have you tried making the payment again to see if the issue resolves itself?
5. Can you please provide me with your order ID or the reference number associated with your payment, if available?

Once I have this information, I'll be able to investigate further and assist you in resolving the issue.

Enter query (or 'exit'): exit


In [10]:
  # ============================================
# Scenario: Basic Research Automation System
# ============================================

# 1. State Definition
# The system maintains a notebook-like state for each research query:
# - topic → The subject to research (e.g., "Quantum Computing").
# - search_results → A list of collected articles or information snippets.
# - analysis → Insights generated from the collected results.
# - summary → Final summarized explanation of the topic.
# - steps_done → Tracks how many steps have been executed.


# ============================================
# 2. Workflow (Graph of States)
# ============================================

# Each research request flows through multiple nodes:

# 🔹 Search Node
# - Simulates web search by generating article titles/snippets
# - Updates:
#   search_results += ["Article 1", "Article 2"]

# 🔹 Analysis Node
# - Counts and reviews collected search results
# - Extracts key insights
# - Updates:
#   analysis = "Key insights from X sources"

# 🔹 Conditional Node (Loop Logic)
# - If number of results < 3:
#       → go back to Search Node (collect more data)
# - Else:
#       → proceed to Summary Node

# 🔹 Summary Node
# - Generates final summary based on analysis
# - Updates:
#   summary = "Summary: Key insights..."

# 🔹 End Node
# - Final summary is returned to the user
 from langgraph.graph import StateGraph, END
from typing import TypedDict

class ResearchState(TypedDict):
    topic: str
    search_results: list
    analysis: str
    summary: str
    steps_done: int

# Node 1: Search for information
def search_web(state: ResearchState):
    print(f"\u001f Searching: {state['topic']}")
    # Simulate web search results
    new_results = [
        f"Article 1 about {state['topic']}",
        f"Article 2 about {state['topic']}",
    ]
    return {"search_results": state["search_results"] + new_results,
            "steps_done": state["steps_done"] + 1}

# Node 2: Analyze the results
def analyze_results(state: ResearchState):
    print(f"\u001f Analyzing {len(state['search_results'])} results")
    analysis = f"Key insights from {len(state['search_results'])} sources"
    return {"analysis": analysis,
            "steps_done": state["steps_done"] + 1}

# Node 3: Summarize
def summarize(state: ResearchState):
    print("\u001f\u001f Generating summary...")
    summary = f"Summary: {state['analysis']}"
    return {"summary": summary}

# Node 4: Check if we need more research
def should_continue(state: ResearchState) -> str:
    if len(state["search_results"]) < 3:
        return "search_web"   # Loop back!
    return END                 # Done

# Build the graph
g = StateGraph(ResearchState)
g.add_node("search_web",  search_web)
g.add_node("analyze",     analyze_results)
g.add_node("summarize",   summarize)

g.set_entry_point("search_web")
g.add_edge("search_web", "analyze")
g.add_conditional_edges("analyze", should_continue,
    {"search_web": "search_web", END: "summarize"})
g.add_edge("summarize", END)

app = g.compile()
result = app.invoke(
    {
        "topic": "Quantum Computing",
        "search_results": [],
        "analysis": "",
        "summary": "",
        "steps_done": 0
    }
)
print(result["summary"])

 Searching: Quantum Computing
 Analyzing 2 results
 Searching: Quantum Computing
 Analyzing 4 results
 Generating summary...
Summary: Key insights from 4 sources


In [11]:
# ============================================
# Scenario: AI Research Assistant (Loop-Based)
# ============================================

# 1. State Definition
# The assistant maintains a notebook-like state for each research task:
# - topic → The subject to research (e.g., "Quantum Computing").
# - search_results → A list of collected information snippets/articles.
# - analysis → Insights extracted from the collected data.
# - summary → Final simplified explanation of the topic.
# - steps_done → Tracks how many steps/iterations are completed.


# ============================================
# 2. Workflow (Graph of States)
# ============================================

# Each research query flows through multiple nodes:

# 🔹 Search Node (HF API)
# - AI generates short research snippets about the topic.
# - Updates:
#   search_results += new snippets

# 🔹 Analysis Node
# - Combines all collected snippets
# - Extracts key insights from them
# - Updates:
#   analysis = summarized insights

# 🔹 Conditional Node (Loop Logic)
# - If collected results are less than 3:
#       → go back to Search Node (gather more data)
# - If enough data is collected:
#       → move to Summary Node

# 🔹 Summary Node
# - Generates a simple, easy-to-understand explanation
# - Updates:
#   summary = final explanation

# 🔹 End Node
# - Final summary is delivered to the user
# Step 1: Install
!pip install huggingface-hub langgraph --quiet

# Step 2: Imports
from langgraph.graph import StateGraph, END
from typing import TypedDict
from huggingface_hub import InferenceClient
from google.colab import userdata

# ============================================
# 🔐 LOAD HF API KEY (Colab Secret: efghi)
# ============================================
hf_key = userdata.get('efghi')

if not hf_key:
    raise ValueError("HF API key not found. Save it as 'efghi' in Colab secrets.")

client = InferenceClient(
    provider="auto",
    api_key=hf_key
)

# ============================================
# 🧠 STATE
# ============================================
class ResearchState(TypedDict):
    topic: str
    search_results: list
    analysis: str
    summary: str
    steps_done: int


# ============================================
# 🤖 HELPER FUNCTION (HF CALL)
# ============================================
def hf_call(prompt: str):
    try:
        response = client.chat.completions.create(
            model="meta-llama/Meta-Llama-3-8B-Instruct",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=200
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"ERROR: {str(e)}"


# ============================================
# 🔎 NODE 1: SEARCH
# ============================================
def search_web(state: ResearchState):
    print(f"🔍 Searching: {state['topic']}")

    new_results = [
        hf_call(f"Give a short article snippet about {state['topic']}"),
        hf_call(f"Give another different snippet about {state['topic']}")
    ]

    return {
        "search_results": state["search_results"] + new_results,
        "steps_done": state["steps_done"] + 1
    }


# ============================================
# 📊 NODE 2: ANALYZE
# ============================================
def analyze_results(state: ResearchState):
    print(f"📊 Analyzing {len(state['search_results'])} results")

    joined = "\n".join(state["search_results"])

    analysis = hf_call(f"""
    Analyze these research snippets and extract key insights:

    {joined}
    """)

    return {
        "analysis": analysis,
        "steps_done": state["steps_done"] + 1
    }


# ============================================
# ✍️ NODE 3: SUMMARIZE
# ============================================
def summarize(state: ResearchState):
    print("✍️ Generating summary...")

    summary = hf_call(f"""
    Summarize this analysis in simple terms:

    {state['analysis']}
    """)

    return {"summary": summary}


# ============================================
# 🔁 LOOP CONTROL
# ============================================
def should_continue(state: ResearchState):
    if len(state["search_results"]) < 3:
        return "search_web"
    return "summarize"


# ============================================
# 🔗 BUILD GRAPH
# ============================================
graph = StateGraph(ResearchState)

graph.add_node("search_web", search_web)
graph.add_node("analyze", analyze_results)
graph.add_node("summarize", summarize)

graph.set_entry_point("search_web")

graph.add_edge("search_web", "analyze")

graph.add_conditional_edges(
    "analyze",
    should_continue,
    {
        "search_web": "search_web",
        "summarize": "summarize"
    }
)

graph.add_edge("summarize", END)


# ============================================
# ▶️ RUN SYSTEM
# ============================================
if __name__ == "__main__":
    app = graph.compile()

    while True:
        topic_input = input("\nEnter topic (or 'exit'): ")

        if topic_input.lower() == "exit":
            break

        state = {
            "topic": topic_input,
            "search_results": [],
            "analysis": "",
            "summary": "",
            "steps_done": 0
        }

        result = app.invoke(state)

        print("\n✅ FINAL SUMMARY:\n", result["summary"])


Enter topic (or 'exit'): artificial intelligence
🔍 Searching: artificial intelligence
📊 Analyzing 2 results
🔍 Searching: artificial intelligence
📊 Analyzing 4 results
✍️ Generating summary...

✅ FINAL SUMMARY:
 Here's a summary of the analysis in simple terms:

Artificial Intelligence (AI) is rapidly changing the way we live and work. It's a technology that allows computers to think and learn like humans, and it's being used in many areas, such as:

- Virtual assistants like Siri and Alexa
- Robotics
- Computer vision (helping computers see and understand the world)
- Predictive analytics (helping predict future outcomes)

However, some people are concerned about how AI makes decisions, so researchers are working on a way to explain these decisions clearly and transparently called Explainable AI (XAI). This is important because it helps us trust the decisions made by AI systems.

Enter topic (or 'exit'): exit


In [12]:
# ============================================
# Scenario: AI Symptom Tracker (Question-Based)
# ============================================

# 1. State Definition
# The assistant maintains a notebook-like state for each patient:
# - symptom → The patient’s reported issue (e.g., "persistent cough").
# - observations → Notes or snippets generated by AI about possible causes or related conditions.
# - analysis → A synthesized interpretation of the observations.
# - recommendation → A simplified, non-medical summary suggesting next steps
#                    (e.g., "consult a doctor if symptoms persist").
# - steps_done → A counter tracking progress through the workflow.


# ============================================
# 2. Workflow (Graph of States)
# ============================================

# Each patient query flows through nodes:

# 🔹 Symptom Input Node
# - Patient reports a symptom
# - State updates:
#   symptom = "persistent cough"

# 🔹 Observation Node (AI API)
# - AI generates possible related factors or general information
# - Updates:
#   observations += new observation

# 🔹 Analysis Node
# - Combines all observations
# - Extracts key insights
# - Updates:
#   analysis = summarized insights

# 🔹 Conditional Node (Loop Logic)
# - If observations < 3:
#       → go back to Observation Node (collect more data)
# - Else:
#       → move to Recommendation Node

# 🔹 Recommendation Node
# - Generates a simple, non-medical suggestion
#   Example:
#   "Seek medical advice if cough lasts more than 2 weeks"
# - Updates:
#   recommendation = final suggestion

# 🔹 End Node
# - Final recommendation is delivered to the patient
# Step 1: Install
!pip install huggingface-hub langgraph --quiet

# Step 2: Imports
from langgraph.graph import StateGraph, END
from typing import TypedDict
from huggingface_hub import InferenceClient
from google.colab import userdata

# ============================================
# 🔐 LOAD HF API KEY (Colab Secret: efghi)
# ============================================
hf_key = userdata.get('efghi')

if not hf_key:
    raise ValueError("HF API key not found. Save it as 'efghi' in Colab secrets.")

client = InferenceClient(
    provider="auto",
    api_key=hf_key
)

# ============================================
# 🧠 STATE
# ============================================
class SymptomState(TypedDict):
    symptom: str
    observations: list
    analysis: str
    recommendation: str
    steps_done: int


# ============================================
# 🤖 HELPER FUNCTION
# ============================================
def hf_call(prompt: str):
    try:
        response = client.chat.completions.create(
            model="meta-llama/Meta-Llama-3-8B-Instruct",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=200
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"ERROR: {str(e)}"


# ============================================
# 🧾 NODE 1: INPUT SYMPTOM
# ============================================
def input_node(state: SymptomState):
    print("\n🩺 Symptom:", state["symptom"])
    return state


# ============================================
# 🔎 NODE 2: OBSERVATION (HF API)
# ============================================
def observation_node(state: SymptomState):
    print("🔍 Collecting observations...")

    new_obs = hf_call(f"""
    Provide one possible general observation or related info about:
    {state['symptom']}
    Keep it short.
    """)

    return {
        "observations": state["observations"] + [new_obs],
        "steps_done": state["steps_done"] + 1
    }


# ============================================
# 🧠 NODE 3: ANALYSIS
# ============================================
def analysis_node(state: SymptomState):
    print("🧠 Analyzing observations...")

    joined = "\n".join(state["observations"])

    analysis = hf_call(f"""
    Analyze these observations and give general insights:

    {joined}
    """)

    return {
        "analysis": analysis,
        "steps_done": state["steps_done"] + 1
    }


# ============================================
# 💡 NODE 4: RECOMMENDATION
# ============================================
def recommendation_node(state: SymptomState):
    print("💡 Generating recommendation...")

    recommendation = hf_call(f"""
    Based on this analysis:

    {state['analysis']}

    Give a simple, non-medical suggestion.
    Avoid diagnosis. Suggest safe next steps.
    """)

    return {
        "recommendation": recommendation
    }


# ============================================
# 🔁 LOOP CONTROL
# ============================================
def should_continue(state: SymptomState):
    if len(state["observations"]) < 3:
        return "observation"
    return "recommendation"


# ============================================
# 🔗 BUILD GRAPH
# ============================================
graph = StateGraph(SymptomState)

graph.add_node("input", input_node)
graph.add_node("observation", observation_node)
graph.add_node("analysis", analysis_node)
graph.add_node("recommendation", recommendation_node)

graph.set_entry_point("input")

graph.add_edge("input", "observation")
graph.add_edge("observation", "analysis")

graph.add_conditional_edges(
    "analysis",
    should_continue,
    {
        "observation": "observation",
        "recommendation": "recommendation"
    }
)

graph.add_edge("recommendation", END)


# ============================================
# ▶️ RUN SYSTEM
# ============================================
if __name__ == "__main__":
    app = graph.compile()

    while True:
        symptom_input = input("\nEnter symptom (or 'exit'): ")

        if symptom_input.lower() == "exit":
            break

        state = {
            "symptom": symptom_input,
            "observations": [],
            "analysis": "",
            "recommendation": "",
            "steps_done": 0
        }

        result = app.invoke(state)

        print("\n✅ FINAL RECOMMENDATION:\n", result["recommendation"])


Enter symptom (or 'exit'): persistent cough

🩺 Symptom: persistent cough
🔍 Collecting observations...
🧠 Analyzing observations...
🔍 Collecting observations...
🧠 Analyzing observations...
🔍 Collecting observations...
🧠 Analyzing observations...
💡 Generating recommendation...

✅ FINAL RECOMMENDATION:
 Based on the analysis, here's a simple, non-medical suggestion:

**Seek Medical Attention**

If you're experiencing a persistent cough, it's essential to consult a healthcare professional for proper evaluation and diagnosis. They will help identify the underlying cause and recommend the best course of treatment.

**Safe Next Steps:**

1. **Schedule an appointment**: Book an appointment with your primary care physician or a pulmonologist.
2. **Keep a cough journal**: Record when your cough occurs, how long it lasts, and any potential triggers (e.g., exposure to smoke, pollen, or dust).
3. **Prepare a list of questions**: Write down any questions or concerns you have, including medical histo

In [15]:
# Scenario: AI-Assisted Email Workflow (Question-Based)
# Context
# A company deploys an AI-powered email assistant to help employees draft, review, and send professional emails.
# The workflow is modeled as a graph of states, where each email task flows through nodes until it is either approved
# and sent or rejected.

# 1. State Definition
# The assistant maintains a notebook-like state:
# - task → The subject or purpose of the email (e.g., "Q3 Report").
# - draft → The AI-generated email draft.
# - approved → A flag indicating whether the human reviewer has approved the draft.

# 2. Workflow (Graph of States)
# Each email task flows through nodes:
# - Draft Node
# - AI generates a draft email based on the task.
# - Updates draft.
# - Review Node (Interrupt)
# - Execution pauses here.
# - Human reviewer inspects the draft and decides whether to approve or reject.
# - Updates approved.
# - Send Node
# - If approved = True → Email is sent.
# - If approved = False → Email is rejected.
# - Updates task with final status (SENT or REJECTED).
# - End Node
# - Workflow completes.

# 3. Example Flow
# - Employee: "Draft an email for the Q3 Report."
# - State: task = "Q3 Report"
# - Assistant drafts:
# Dear User,
# Regarding: Q3 Report
# [AI drafted content]
# - Human reviews → Approves draft (approved = True)
# - Assistant sends → task = "SENT: Q3 Report"
# - Final Output: ✅ Email delivered.

# 👉 Scenario Question:
# "Imagine you are designing an AI-powered email assistant that drafts emails, pauses for human review, and
# then either sends or rejects them. How would you structure the state and workflow graph to ensure accountability
#  and human oversight in the process?"
# Pause before review
# Step 1: Install
!pip install huggingface-hub langgraph --quiet

# Step 2: Imports
from langgraph.graph import StateGraph, END
from typing import TypedDict
from huggingface_hub import InferenceClient
from google.colab import userdata

# ============================================
# 🔐 LOAD HF API KEY
# ============================================
hf_key = userdata.get('efghi')

if not hf_key:
    raise ValueError("HF API key not found. Save it as 'efghi' in Colab secrets.")

client = InferenceClient(provider="auto", api_key=hf_key)

# ============================================
# 🧠 STATE
# ============================================
class AIApprovalState(TypedDict):
    task: str
    ai_output: str
    risk_level: str
    reviewed_output: str
    decision: str
    feedback: str


# ============================================
# 🤖 HELPER FUNCTION
# ============================================
def hf_call(prompt):
    try:
        response = client.chat.completions.create(
            model="meta-llama/Meta-Llama-3-8B-Instruct",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=200
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"ERROR: {str(e)}"


# ============================================
# 🧾 NODE 1: GENERATE AI OUTPUT
# ============================================
def generate_ai_output(state: AIApprovalState):
    print("🤖 Generating AI output...")

    output = hf_call(f"""
    Perform this task and generate output:
    {state['task']}
    """)

    return {"ai_output": output}


# ============================================
# ⚠️ NODE 2: RISK CHECK
# ============================================
def risk_check(state: AIApprovalState):
    print("⚠️ Checking risk level...")

    risk = hf_call(f"""
    Task: {state['task']}
    Output: {state['ai_output']}

    Classify risk:
    low / medium / high / critical
    """)

    return {"risk_level": risk.lower()}


# ============================================
# 👨‍⚖️ NODE 3: HUMAN REVIEW (SIMULATED)
# ============================================
def human_review(state: AIApprovalState):
    print("👨‍⚖️ Human review required...")

    reviewed = input(f"\nEdit/Approve Output:\n{state['ai_output']}\n\nEnter edited version: ")

    return {"reviewed_output": reviewed}


# ============================================
# 🧠 NODE 4: FINAL DECISION
# ============================================
def decision_node(state: AIApprovalState):
    print("🧠 Making final decision...")

    decision = hf_call(f"""
    Risk: {state['risk_level']}
    Reviewed Output: {state['reviewed_output']}

    Decide:
    approve / reject
    """)

    return {"decision": decision.lower()}


# ============================================
# 🔁 NODE 5: FEEDBACK LOOP
# ============================================
def feedback_node(state: AIApprovalState):
    print("🔁 Generating feedback...")

    feedback = hf_call(f"""
    Original Output: {state['ai_output']}
    Final Output: {state['reviewed_output']}

    Give improvement feedback for AI.
    """)

    return {"feedback": feedback}


# ============================================
# 🔀 ROUTING (HUMAN-IN-LOOP)
# ============================================
def route_based_on_risk(state: AIApprovalState):
    if "high" in state["risk_level"] or "critical" in state["risk_level"]:
        return "review"
    return "decision"


# ============================================
# 🔗 BUILD GRAPH
# ============================================
graph = StateGraph(AIApprovalState)

graph.add_node("generate", generate_ai_output)
graph.add_node("risk", risk_check)
graph.add_node("review", human_review)
graph.add_node("decision", decision_node)
graph.add_node("feedback", feedback_node)

graph.set_entry_point("generate")

graph.add_edge("generate", "risk")

graph.add_conditional_edges(
    "risk",
    route_based_on_risk,
    {
        "review": "review",
        "decision": "decision"
    }
)

graph.add_edge("review", "decision")
graph.add_edge("decision", "feedback")
graph.add_edge("feedback", END)


# ============================================
# ▶️ RUN SYSTEM
# ============================================
if __name__ == "__main__":
    app = graph.compile()

    while True:
        task_input = input("\nEnter AI task (or 'exit'): ")

        if task_input.lower() == "exit":
            break

        state = {
            "task": task_input,
            "ai_output": "",
            "risk_level": "",
            "reviewed_output": "",
            "decision": "",
            "feedback": ""
        }

        result = app.invoke(state)

        print("\n✅ FINAL DECISION:", result["decision"])
        print("\n📝 FEEDBACK:", result["feedback"])


Enter AI task (or 'exit'): Write a marketing email for new product
🤖 Generating AI output...
⚠️ Checking risk level...
🧠 Making final decision...
🔁 Generating feedback...

✅ FINAL DECISION: approved 

reasoning: 
1. the email is from a legitimate company introducing a new product, which suggests a genuine marketing purpose.
2. the potential risks mentioned (spam filter evasion, data collection, unsolicited communication) are common issues with marketing emails.
3. the suggested mitigation steps (verify recipient consent, use clear and transparent language) are reasonable and align with best practices for marketing emails.

📝 FEEDBACK: **Feedback for AI Model: Improving the Output**

The original output is a promotional email for the SmartLife Device, a smart home assistant. While the content is informative and engaging, there are areas where the AI model could improve for better output.

**Improvement Suggestions:**

1.  **Personalization:** The email address the customer by their nam

In [ ]:
#Imagine you are designing an AI-powered assistant that drafts structured reports, pauses for human review, and then either publishes or rejects them.
#How would you structure the state and workflow graph to ensure accountability and human oversight in the process?
# Step 1: Install
!pip install huggingface-hub langgraph --quiet

# Step 2: Imports
from langgraph.graph import StateGraph, END
from typing import TypedDict
from huggingface_hub import InferenceClient
from google.colab import userdata
from datetime import datetime

# ============================================
# 🔐 LOAD HF API KEY
# ============================================
hf_key = userdata.get('efghi')

if not hf_key:
    raise ValueError("HF API key not found. Save it as 'efghi' in Colab secrets.")

client = InferenceClient(
    provider="auto",
    api_key=hf_key
)

# ============================================
# 🧠 STATE
# ============================================
class ReportState(TypedDict):
    topic: str
    draft: str
    review_status: str
    feedback: str
    final_output: str
    log: list


# ============================================
# ✍️ NODE 1: GENERATE DRAFT
# ============================================
def generate_draft(state: ReportState):
    topic = state["topic"]

    try:
        response = client.chat.completions.create(
            model="meta-llama/Meta-Llama-3-8B-Instruct",
            messages=[{
                "role": "user",
                "content": f"""
                Write a structured report on: {topic}
                Include introduction, key points, and conclusion.
                """
            }],
            max_tokens=300
        )

        draft = response.choices[0].message.content

    except Exception as e:
        draft = f"Error: {str(e)}"

    print("\n📄 Draft Generated:\n", draft)

    return {"draft": draft}


# ============================================
# 👤 NODE 2: HUMAN REVIEW
# ============================================
def human_review(state: ReportState):
    print("\n🔍 Review the draft above.")

    decision = input("👉 Approve or Reject? ").lower()
    feedback = ""

    if decision == "reject":
        feedback = input("👉 Enter feedback: ")

    return {
        "review_status": decision,
        "feedback": feedback
    }


# ============================================
# 🔁 NODE 3: IMPROVE DRAFT
# ============================================
def improve_draft(state: ReportState):
    try:
        response = client.chat.completions.create(
            model="meta-llama/Meta-Llama-3-8B-Instruct",
            messages=[{
                "role": "user",
                "content": f"""
                Improve this report based on feedback:

                Report:
                {state['draft']}

                Feedback:
                {state['feedback']}
                """
            }],
            max_tokens=300
        )

        new_draft = response.choices[0].message.content

    except Exception as e:
        new_draft = f"Error: {str(e)}"

    print("\n✍️ Improved Draft:\n", new_draft)

    return {"draft": new_draft}


# ============================================
# 📢 NODE 4: PUBLISH
# ============================================
def publish_report(state: ReportState):
    print("\n✅ Report Approved & Published!")

    return {
        "final_output": state["draft"]
    }


# ============================================
# 🧾 NODE 5: LOGGING
# ============================================
def log_step(state: ReportState):
    entry = {
        "topic": state["topic"],
        "status": state["review_status"],
        "time": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    return {
        "log": state["log"] + [entry]
    }


# ============================================
# 🔁 CONDITION
# ============================================
def review_decision(state: ReportState):
    if state["review_status"] == "approved":
        return "publish"
    return "improve"


# ============================================
# 🔗 BUILD GRAPH
# ============================================
graph = StateGraph(ReportState)

graph.add_node("generate", generate_draft)
graph.add_node("review", human_review)
graph.add_node("improve", improve_draft)
graph.add_node("publish", publish_report)
graph.add_node("log", log_step)

graph.set_entry_point("generate")

graph.add_edge("generate", "review")

graph.add_conditional_edges(
    "review",
    review_decision,
    {
        "publish": "publish",
        "improve": "improve"
    }
)

graph.add_edge("improve", "review")   # Loop until approved
graph.add_edge("publish", "log")
graph.add_edge("log", END)


# ============================================
# ▶️ RUN SYSTEM
# ============================================
if __name__ == "__main__":
    app = graph.compile()

    report_log = []

    while True:
        topic_input = input("\nEnter report topic (or 'exit'): ")

        if topic_input.lower() == "exit":
            break

        state = {
            "topic": topic_input,
            "draft": "",
            "review_status": "",
            "feedback": "",
            "final_output": "",
            "log": report_log
        }

        result = app.invoke(state)

        report_log = result["log"]

        print("\n📚 Final Report:\n", result["final_output"])


Enter report topic (or 'exit'): Do you offer refunds or returns?

📄 Draft Generated:
 **Return and Refund Policy Report**

**Introduction**

At [Company Name], we value our customers and strive to provide the best possible shopping experience. As part of our commitment to customer satisfaction, we have established a clear return and refund policy to ensure that customers can shop with confidence. This report outlines the key points of our return and refund policy.

**Key Points**

### 1. Eligibility for Returns and Refunds

* Customers are eligible for returns and refunds within [timeframe, e.g., 30 days] of receiving their order.
* Items must be in their original condition, with tags and packaging intact.
* Products that have been worn, used, or damaged are not eligible for returns or refunds.

### 2. Return Process

* Customers can initiate the return process by contacting our customer service team via phone, email, or through our website.
* Customers must provide a valid reason for